# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alex-jb/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2 — Refresh / Content Opportunity Scoring. Task type: RANKING / SCORING, built on a binary CLASSIFICATION target.**

The decision is *"which pages should an editor review first?"* — that's a **ranking** problem, judged at the top of the list, not a yes/no verdict on every page. Under the hood I predict a **per-page probability of decline** (a classification model), then **rank pages by that probability** to produce the refresh queue. So the pipeline is *classify → probability → rank*.

It is **not clustering** (I have a target), and it is **not plain classification** (I care about the *order of the top 50*, not global accuracy on the 30k middle nobody will touch).

In [1]:
import os, sys, pandas as pd, numpy as np
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship"):
        os.system("git clone --depth 1 https://github.com/alex-jb/flyrank-ml-internship flyrank-ml-internship")
    os.chdir("flyrank-ml-internship")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)   # proxy target (see section 2)
print(f"{len(df):,} pages | {df['client_id'].nunique()} clients")
print(f"proxy decline base rate: {df['is_declining_label'].mean():.3f}")
print("task = RANK pages by predicted decline probability  (classification model -> ranked queue)")

30,000 pages | 32 clients
proxy decline base rate: 0.542
task = RANK pages by predicted decline probability  (classification model -> ranked queue)


## 2. Target or proxy

**Proxy I can use today:** `is_declining_label = (trend_direction == "down")`.

**Where it comes from — honestly:** this label is **defined by a rule**, not observed. It's derived from `trend_direction` (they agree 100%, shown below), which is computed from `trend_pct`. So a model trained on it just relearns FlyRank's own rule — and `trend_direction` / `trend_pct` can **never** be features (that's leakage).

**The target I actually want (capstone):** an **observed future-window outcome** — did a page's traffic actually drop in a *later* month, measured against its own prior window (from the warehouse `fact_content_daily_performance`)? That's *measured, not defined*, and it's what makes the model predict the world instead of a formula. For this starter notebook I sketch the proxy; the capstone swaps in the observed label with a leakage audit.

In [2]:
agree = (df["is_declining_label"] == df["trend_direction"].str.lower().eq("down").astype(int)).mean()
print(f"proxy label == (trend_direction=='down') agreement: {agree:.3f}  -> rule-defined, NOT measured")
print("label counts:", df["is_declining_label"].value_counts().to_dict())
print("LEAKAGE columns — never features:", ["trend_direction", "trend_pct"])
print("Capstone target instead: an OBSERVED drop in a FUTURE window (warehouse), measured not defined.")

proxy label == (trend_direction=='down') agreement: 1.000  -> rule-defined, NOT measured
label counts: {1: 16262, 0: 13738}
LEAKAGE columns — never features: ['trend_direction', 'trend_pct']
Capstone target instead: an OBSERVED drop in a FUTURE window (warehouse), measured not defined.


## 3. Success metric

**Primary: precision@K (precision@50).** An editor only works the *top* of the queue, so what matters is *how many of the top 50 pages are genuinely declining*, not global accuracy — accuracy rewards being right about the huge "fine" middle that nobody will act on.

**What "good" means (a defensible bar):** beat two references by a wide margin — (1) the base rate ≈ **0.54** ("always guess declining"), and (2) the hand-rule baseline, which ML-02 measured at **0.240** precision@50 on client-holdout. ML-02's learned model already hit **0.740** (~3×). Secondary metric: **ROC-AUC** for overall separation. Everything validated with **client-holdout** so a client never appears in both train and test.

In [3]:
base = df["is_declining_label"].mean()
print(f"'always guess declining' precision ~= base rate = {base:.3f}   <- the floor to beat")
print("From ML-02 (client-holdout, scripts/run_all.py):")
print("  precision@50:  hand rule 0.240  ->  random forest 0.740   (~3.1x)")
print("Defended metric: precision@50 (top-of-queue). Secondary: ROC-AUC. Validation: client-holdout.")

'always guess declining' precision ~= base rate = 0.542   <- the floor to beat
From ML-02 (client-holdout, scripts/run_all.py):
  precision@50:  hand rule 0.240  ->  random forest 0.740   (~3.1x)
Defended metric: precision@50 (top-of-queue). Secondary: ROC-AUC. Validation: client-holdout.


## 4. The unit of analysis, as a real dataframe

**One row = one page (content item).** Identified by `content_id`, grouped by `client_id` (32 clients). The grain is verified below (rows == unique `content_id`). `client_id` is a **grouping key for train/test splits, never a feature** (it's a pseudonym). Each page carries its trailing-90-day metrics; the score will rank these rows.

In [4]:
print("grain check - one row per page (rows == unique content_id):", len(df) == df["content_id"].nunique())
cols = ["content_id", "client_id", "impressions_90d", "avg_position",
        "days_since_last_update", "ctr", "trend_direction", "is_declining_label"]
sample = df[cols].head(6)
sample

grain check - one row per page (rows == unique content_id): True


,content_id,client_id,impressions_90d,avg_position,days_since_last_update,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,20,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,25,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,20,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,22,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,14,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,20,0.03,down,1


## 5. Why ML beats a fixed rule here

No single feature separates decline cleanly, and some relationships are **non-monotonic** — which is exactly where an if-statement breaks:

- **`position_tier` (the giveaway):** decline is **low at both extremes** (`top_3` ≈ 0.24, `deep` ≈ 0.34) but **high in the middle** (`page_1` ≈ 0.57, `striking` ≈ 0.61). A rule like *"worse position → refresh"* gets `deep` pages backwards.
- **`word_count_tier`:** only the very short pages (`<1000` ≈ 0.21) stand out; everything above is a flat ~0.56–0.60 — length helps only at the bottom.
- **`age_tier`:** monotonic the *other* way (young `31-90` ≈ 0.67 → old `365+` ≈ 0.43).

Each signal is weak and partial, and decline lives in their **interactions** (a stale, mid-position, high-demand page is a different animal from a fresh top-3 page). A fixed rule can encode *one* threshold on *one* feature; it can't weigh six interacting signals with opposite-signed effects. That's the messy-but-real pattern ML is for — and ML-02 **measured** the payoff: hand rule 0.240 → learned 0.740 precision@50.

In [5]:
for col in ["position_tier", "word_count_tier", "age_tier"]:
    r = df.groupby(col)["is_declining_label"].mean().round(3)
    print(f"decline rate by {col}:")
    print(r.to_string()); print()
print("position_tier is non-monotonic: LOW at top_3 & deep, HIGH in the middle")
print("-> 'worse position => refresh' is backwards for deep pages. Needs interactions, not one threshold.")
print("Measured payoff (ML-02, client-holdout): hand rule 0.240 -> learned 0.740 precision@50.")

decline rate by position_tier:
position_tier
deep        0.344
page_1      0.570
page_3_5    0.562
striking    0.610
top_3       0.241

decline rate by word_count_tier:
word_count_tier
1000-2000    0.556
2000-3500    0.588
3500+        0.597
<1000        0.207

decline rate by age_tier:
age_tier
181-365    0.515
31-90      0.669
365+       0.426
91-180     0.626

position_tier is non-monotonic: LOW at top_3 & deep, HIGH in the middle
-> 'worse position => refresh' is backwards for deep pages. Needs interactions, not one threshold.
Measured payoff (ML-02, client-holdout): hand rule 0.240 -> learned 0.740 precision@50.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.